[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/surfaces/k_surfaces/k_surfaces.ipynb)

# Discrete Saddle Surfaces

This example grows discrete surfaces of constant negative curvature and their Bäcklund transformations from two crossing curves of unit normals. A surface of constant negative curvature bends one way along one family of lines and the opposite way across it, forming a saddle at every point, and on a discrete quadrilateral net three corner normals determine the fourth while preserving every edge length.

Each surface normal is modelled as an extrinsic `Plane`, and reflecting a quadrilateral's base normal across the bisector `(left - right).normalized() >> base` completes the missing corner normal. Neighboring tangent planes meet along the intrinsic displacement `radius * Plane.commutator(normals)(neighbour)`, scaled and oriented by the volume element `radius` so reflecting the ambient frame reflects every vertex automatically. Propagating the same reflection rule from a single new normal at one corner builds a Bäcklund surface joined to the original sheet by equal-length segments tangent to both sheets.

In Gibbs-Heaviside vector notation on $\mathbb{R}^3$, as in Bobenko and Suris, [*Discrete Differential Geometry: Consistency as Integrability*, sections 2.4-2.5](https://arxiv.org/pdf/math/0504358v1#page=58), completing a Moutard normal quadrilateral expands the Householder reflection into $\mathbf{n}_{12} = \frac{2\,\mathbf{n}_0\cdot(\mathbf{n}_1 + \mathbf{n}_2)}{|\mathbf{n}_1 + \mathbf{n}_2|^2}(\mathbf{n}_1 + \mathbf{n}_2) - \mathbf{n}_0$, while reconstructing vertex positions via the Lelieuvre cross-product differences $\mathbf{f}_1 - \mathbf{f}_0 = R\,(\mathbf{n}_1 \times \mathbf{n}_0)$ introduces a right-hand-rule sign that reverses under mirror reflections unless multiplied by $\det M$.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ https://github.com/EelcoHoogendoorn/numga/archive/HEAD.tar.gz"

In [ ]:
from collections.abc import Callable, Iterator

import numpy as np

from numga import NumpyContext, concatenate, stack
from numga.algebras import VGA3D as ga
from examples.util.animation import show_animation
from examples.surfaces.k_surfaces import render

mv = NumpyContext(ga).multivector
# Tangent-plane normals are extrinsic planes; surface vertices and edge displacements are intrinsic points.
Plane = ga.gatype.vector()                          # Vector+
Point = ga.gatype.antivector()                      # Antivector-
Rotor = ga.gatype.rotor()                           # Rotor+
# Intrinsic volume element giving the radius of curvature its orientation under reflections.
Antiscalar = ga.gatype.antipseudoscalar()           # Antipseudoscalar-

## Bending a Chebyshev saddle net

Varying the intersection angle between the two central normal curves deforms the saddle in space while keeping every quadrilateral edge at the same length; only the interior angles and diagonals shear. Because each diagonal of the grid depends only on the two preceding diagonals, the reflection rule fills all four quadrants outwards from the central cross.

In [ ]:
# Half-grid vertex count, normal-arc angular span, curvature radius, and corner-angle bending sweep.
n_half = 12
n_vertices = 2 * n_half + 1
extent = 0.8
curvature_radius = 1
neutral_angle = np.pi / 2
bend_amplitude = np.pi / 6
n_frames = 84
n_bend_states = n_frames // 2 + 1
frame_duration_ms = 40
phases = np.linspace(-np.pi / 2, np.pi / 2, n_bend_states)
corner_angles = neutral_angle + bend_amplitude * np.sin(phases)
corner_extremes = neutral_angle + bend_amplitude * np.array([-1, 1])
radius = curvature_radius * mv.xyz.dual()                           # [] Antiscalar-

In [ ]:
def quad_normal(base: Plane, left: Plane, right: Plane) -> Plane:
    """Reflecting `base` across the bisector plane `(left - right).normalized()` completes a compatible quadrilateral of unit normals (`[...] Plane+`)."""
    bisector = (left - right).normalized()                                      # [...] Plane+
    return bisector >> base                                                     # [...] Plane+


def surface_points(normals: Plane, radius: Antiscalar, origin: Point) -> Point:
    """Summing tangent-plane intersection edges `radius * Plane.commutator(normals)(neighbour)` from `origin` reconstructs the K-surface vertices (`[..., n_u, n_v] Point-`)."""
    n_u, n_v = normals.shape[-2:]
    del n_u, n_v
    # Each normal maps a neighboring normal to the displacement along the line where their tangent planes meet.
    edge_map = radius[..., None, None] * Plane.commutator(normals)              # [..., n_u, n_v] Point- <- Plane+
    edges_u = edge_map[..., :-1, :](normals[..., 1:, :])                        # [..., n_u - 1, n_v] Point-
    edges_v = edge_map[..., :, :-1](normals[..., :, 1:])                        # [..., n_u, n_v - 1] Point-
    first = origin + edges_u[..., 0, 0] * 0                                     # [...] Point-
    row_origins = concatenate([first[..., None], first[..., None] + edges_u[..., :, 0].cumsum(axis=-1)], axis=-1)
    return row_origins[..., :, None] + concatenate([edges_v[..., :, :1] * 0, edges_v.cumsum(axis=-1)], axis=-1)

In [ ]:
def normal_net(mid_u: Plane, mid_v: Plane, completion: Callable[[Plane, Plane, Plane], Plane]) -> Plane:
    """Two crossing odd-length normal curves `mid_u` (`[..., n_u] Plane+`) and `mid_v` (`[..., n_v] Plane+`) propagate diagonally across the `[..., n_u, n_v]` net."""
    n_u, n_v = mid_u.shape[-1], mid_v.shape[-1]
    n_half_u, n_half_v = n_u // 2, n_v // 2
    crossing = mid_u[..., n_half_u] + mid_v[..., n_half_v]                      # [...] Plane+
    normals = crossing[..., None, None] * np.zeros((n_u, n_v))                  # [..., n_u, n_v] Plane+
    normals = normals.at[..., :, n_half_v].set(mid_u)
    normals = normals.at[..., n_half_u, :].set(mid_v)
    directions_u = np.array([1, 1, -1, -1])[:, None]
    directions_v = np.array([1, -1, 1, -1])[:, None]
    # Each diagonal in all four quadrants depends only on the two preceding diagonals.
    for diagonal in range(2, n_half_u + n_half_v + 1):
        distance_u = np.arange(max(1, diagonal - n_half_v), min(n_half_u, diagonal - 1) + 1)
        distance_v = diagonal - distance_u
        row = n_half_u + directions_u * distance_u
        column = n_half_v + directions_v * distance_v
        previous_row = row - directions_u
        previous_column = column - directions_v
        normals = normals.at[..., row, column].set(completion(
            normals[..., previous_row, previous_column],
            normals[..., row, previous_column],
            normals[..., previous_row, column]))
    return normals


def saddle_patch(n_half: int, extent: float, corner_angle: float, radius: Antiscalar) -> tuple[Plane, Point]:
    """Two great-circle normal arcs meeting at `corner_angle` generate a centred K-surface patch (`[n_vertices, n_vertices] Plane+`, `[n_vertices, n_vertices] Point-`)."""
    n_vertices = 2 * n_half + 1
    angles = np.linspace(-extent, extent, n_vertices)                           # [n_vertices]
    alternating = 1 - 2 * ((np.arange(n_vertices) - n_half) % 2)                # [n_vertices]
    reference_normal = mv.vector([0, 0, 1])                                     # [] Plane+
    corner_turn = (mv.xy * (-corner_angle / 2)).exp()                           # [] Rotor+
    second_plane = corner_turn >> mv.yz                                         # [] Bivector+
    turns_u = (mv.yz * (-angles / 2)).exp()                                     # [n_vertices] Rotor+
    mid_u = turns_u >> reference_normal                                         # [n_vertices] Plane+
    turns_v = (second_plane * (-angles / 2)).exp()                              # [n_vertices] Rotor+
    # Alternating signs on adjacent columns allow one reflection rule to serve both grid directions.
    mid_v = alternating * (turns_v >> reference_normal)                         # [n_vertices] Plane+
    normals = normal_net(mid_u, mid_v, quad_normal)                             # [n_vertices, n_vertices] Plane+
    points = surface_points(normals, radius, reference_normal.dual() * 0)       # [n_vertices, n_vertices] Point-
    return normals, points - points[n_half, n_half]

In [ ]:
def saddle_states(n_half: int, extent: float, corner_angles: np.ndarray, radius: Antiscalar) -> Iterator[Point]:
    """Yields centred saddle vertex nets (`[n_vertices, n_vertices] Point-`) as the central normal-arc angle sweeps `corner_angles`."""
    for corner_angle in corner_angles:
        normals, points = saddle_patch(n_half, extent, float(corner_angle), radius)
        del normals
        yield points


reference = stack(list(saddle_states(n_half, extent, corner_extremes, radius))) # [2, n_vertices, n_vertices] Point-
recorded_saddles: list[Point] = []


def streamed_saddles() -> Iterator[Point]:
    """Yields each bent saddle vertex net while recording it once for downstream edge-length checks."""
    for state in saddle_states(n_half, extent, corner_angles, radius):
        recorded_saddles.append(state)
        yield state


show_animation(
    render.saddle_animation(reference, streamed_saddles()),
    "k_surfaces_saddle",
    frame_duration_ms,
    n_frames=n_bend_states,
    forward_then_back=True,
);

Teal marks the front face of the gridded saddle and copper marks its underside as the net shears smoothly between acute and obtuse central crossings under a fixed orthographic camera.

## Bäcklund transformation along common tangents

Choosing a single new normal at one corner of the saddle and marching the quadrilateral reflection rule across the rows and columns generates a second K-surface. Corresponding vertices on the two sheets are displaced by `radius * transformed_normals.commutator(normals)`, giving segments of uniform length that lie simultaneously in the tangent planes of both surfaces. Rendering the orbiting pair takes about ten seconds.

In [ ]:
# Corner normal seed angles and camera orbit sweep.
seed_azimuth = 1.6
seed_turn = 0.65
n_orbit_frames = 180
n_orbit_views = n_orbit_frames // 2 + 1
orbit_elevation = 26
orbit_azimuth_centre = -70
orbit_azimuth_amplitude = 50
orbit_phases = np.linspace(0, np.pi, n_orbit_views)
orbit_azimuths = orbit_azimuth_centre + orbit_azimuth_amplitude * np.cos(orbit_phases)

In [ ]:
def backlund_normals(normals: Plane, start: Plane, completion: Callable[[Plane, Plane, Plane], Plane]) -> Plane:
    """Propagating `completion` from `start` at corner `(0, 0)` builds a compatible Bäcklund normal net (`[..., n_u, n_v] Plane+`)."""
    n_u, n_v = normals.shape[-2:]
    result = (normals + start[..., None, None]) * 0                             # [..., n_u, n_v] Plane+
    result = result.at[..., 0, 0].set(start)
    for row in range(1, n_u):
        result = result.at[..., row, 0].set(completion(
            normals[..., row - 1, 0], normals[..., row, 0], result[..., row - 1, 0]))
    # Once the first column is determined, every row advances in lockstep across the columns.
    for column in range(1, n_v):
        result = result.at[..., :, column].set(completion(
            normals[..., :, column - 1], normals[..., :, column], result[..., :, column - 1]))
    return result


normals, points = saddle_patch(n_half, extent, neutral_angle, radius)           # [n_vertices, n_vertices] Plane+, Point-
corner_normal = normals[0, 0]                                                   # [] Plane+
reference_plane = (corner_normal ^ mv.x).normalized()                           # [] Bivector+
spin_plane = corner_normal | mv.xyz                                             # [] Bivector+
azimuth_turn = (spin_plane * (-seed_azimuth / 2)).exp()                         # [] Rotor+
seed_plane = azimuth_turn >> reference_plane                                    # [] Bivector+
elevation_turn = (seed_plane * (-seed_turn / 2)).exp()                          # [] Rotor+
starting_normal = elevation_turn >> corner_normal                               # [] Plane+

transformed_normals = backlund_normals(normals, starting_normal, quad_normal)   # [n_vertices, n_vertices] Plane+
edge_map = radius * Plane.commutator(normals)                                   # [n_vertices, n_vertices] Point- <- Plane+
transformed_points = points + edge_map(transformed_normals)                     # [n_vertices, n_vertices] Point-

show_animation(
    render.backlund_orbit_animation(transformed_points, points, orbit_azimuths, orbit_elevation),
    "k_surfaces_backlund",
    frame_duration_ms,
    n_frames=n_orbit_views,
    forward_then_back=True,
);

Dark rods with spherical endpoints join matching grid vertices on the translucent teal source saddle and copper Bäcklund sheet, grazing both sheets tangentially while showing faintly through occluding faces as the camera orbits.

## Bianchi permutability

Applying two distinct Bäcklund transformations to the same saddle and closing every normal cube with `quad_normal` yields a fourth K-surface whose vertices coincide whichever intermediate sheet is traversed.

In [ ]:
seed_turns = np.array([0.25, 0.7])

In [ ]:
pair_turns = (seed_plane * (-seed_turns / 2)).exp()                             # [2] Rotor+
starting_normals = pair_turns >> corner_normal                                  # [2] Plane+
first_normals, second_normals = backlund_normals(normals, starting_normals, quad_normal)
common_normals = quad_normal(normals, first_normals, second_normals)            # [n_vertices, n_vertices] Plane+
first_points = points + edge_map(first_normals)                                 # [n_vertices, n_vertices] Point-
second_points = points + edge_map(second_normals)                               # [n_vertices, n_vertices] Point-
via_first = first_points + radius * common_normals.commutator(first_normals)    # [n_vertices, n_vertices] Point-
via_second = second_points + radius * common_normals.commutator(second_normals) # [n_vertices, n_vertices] Point-

In [ ]:
# checks
assert len(recorded_saddles) == n_bend_states
# Bending preserves every grid edge length across the deformation.
bent_patches = stack([recorded_saddles[0], recorded_saddles[-1]])               # [2, n_vertices, n_vertices] Point-
edge_lengths = (bent_patches[:, 1:, :] - bent_patches[:, :-1, :]).antinorm()    # [2, n_vertices - 1, n_vertices] Scalar+
assert ((edge_lengths - edge_lengths[0, 0, 0]).abs() < 1e-10).all()
# Bianchi completion reaches the same fourth surface along both routes, and Bäcklund connectors are equal and tangent.
assert ((via_first - via_second).antinorm() < 1e-10).all()
connectors = transformed_points - points                                        # [n_vertices, n_vertices] Point-
assert ((connectors.antinorm() - connectors[0, 0].antinorm()).abs() < 1e-10).all()
assert ((normals & connectors).abs() < 1e-10).all()
assert ((transformed_normals & connectors).abs() < 1e-10).all()